# PEfE: evolve foraging agents

Clones the repository, evolves a population of sparse recurrent-network agents,
evaluates the best genome on 20 held-out episodes (plus the *blind* check with
food sensors zeroed), and renders the best team. Pure numpy, no GPU or torch needed.

Set `BRANCH` to test a branch before it is merged. `GENERATIONS = 150` is the
full default run (about 2 minutes); `SEED = 0, GENERATIONS = 40` reproduces the
reference check in the README (best validation 15.82, score 15.62 ± 0.83, blind 0.23, 82 connections).

In [ ]:
import os
BRANCH = os.environ.get("PEFE_BRANCH", "main")
REPO_URL = os.environ.get("PEFE_REPO_URL", "https://github.com/PrimozRavbar/PEfE_exp.git")
SEED = int(os.environ.get("PEFE_SEED", 0))
GENERATIONS = int(os.environ.get("PEFE_GENERATIONS", 150))
RUN = f"runs/seed{SEED}_gen{GENERATIONS}"

In [ ]:
# clone (fresh copy each time this cell runs)
import shutil
REPO_DIR = os.path.abspath("PEfE_exp")
shutil.rmtree(REPO_DIR, ignore_errors=True)
!git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!pip -q install numpy matplotlib
if shutil.which("ffmpeg") is None:
    !apt-get -qq install -y ffmpeg > /dev/null
!git log --oneline -1

In [ ]:
# evolve (all hyperparameters are the evolve.py defaults; see README)
!python src/evolve.py --out {RUN} --seed {SEED} --generations {GENERATIONS}

In [ ]:
# evaluate: 20 held-out episodes, and the same with food sensors zeroed (blind)
!python src/evaluate.py --genome {RUN}/best.npz --json {RUN}/eval.json
import json
r = json.load(open(f"{RUN}/eval.json"))[0]
print(f"best validation {r['meta']['valid']:.2f}")
print(f"score {r['score']['mean']:.2f} ± {r['score']['sd']:.2f}   "
      f"blind {r['blind']['mean']:.2f}   connections {r['connections']}")

In [ ]:
# render the best clone team on the first held-out seed
!python src/render.py --genome {RUN}/best.npz --out {RUN}/best.mp4
from IPython.display import Image, Video, display
display(Image(f"{RUN}/best_trails.png"))
display(Video(f"{RUN}/best.mp4", embed=True, width=480))